# 🚦 Phase 4: Exploratory Data Analysis (EDA)
### Traffic Sign Recognition using Deep Learning & CNN

---

## 🎯 Objectives for Phase 4:
1. **Audit Dataset Structure**: Scan all 43 folders (`0` through `42`) in `dataset/gtsrb` and count available images.
2. **Analyze Class Distribution**: Compute key statistics (min, max, mean, median) and visualize class frequencies to identify **class imbalance**.
3. **Inspect Image Dimensions**: Examine original image resolutions and aspect ratios to understand why standardizing to $32 \times 32$ pixels is required.
4. **Visual Gallery of All 43 Signs**: Display a comprehensive grid showing one representative sample from each traffic sign category.
5. **Real-World Variations**: Inspect real-world challenges (lighting conditions, shadows, motion blur, and angles) within the same sign class.

> **Beginner Tip:** Simply run each cell sequentially using `Shift + Enter`.

### 📁 Step 4.1: Import Libraries & Configure Paths

In [ ]:
import os
import sys
import cv2
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

# Handle path when running inside notebooks folder or project root
if os.path.basename(os.getcwd()) == "notebooks":
    PROJECT_ROOT = os.path.abspath("..")
else:
    PROJECT_ROOT = os.path.abspath(".")

sys.path.append(PROJECT_ROOT)
from label_utils import CLASS_NAMES, SIGN_CATEGORIES, get_full_metadata

DATASET_PATH = os.path.join(PROJECT_ROOT, "dataset", "gtsrb")

print("Project Root :", PROJECT_ROOT)
print("Dataset Path :", DATASET_PATH)

if not os.path.exists(DATASET_PATH):
    print("\n⚠️ Dataset not found at path! Please run Phase 2 notebook (02_phase2_dataset_acquisition.ipynb) first.")
else:
    print("✅ Dataset folder verified!")

### 📊 Step 4.2: Scan Dataset & Compute Class Statistics
Let's count how many images exist in each folder from class `0` to `42`.

In [ ]:
class_counts = {}
sample_files = {}
all_image_paths = []

for cid in range(43):
    folder = os.path.join(DATASET_PATH, str(cid))
    if os.path.exists(folder):
        files = [f for f in os.listdir(folder) if f.lower().endswith(('.ppm', '.png', '.jpg', '.jpeg'))]
        class_counts[cid] = len(files)
        if files:
            sample_files[cid] = os.path.join(folder, files[0])
            for f in files:
                all_image_paths.append((cid, os.path.join(folder, f)))
    else:
        class_counts[cid] = 0

counts_list = list(class_counts.values())
total_images = sum(counts_list)

print("=" * 50)
print("📈 DATASET SUMMARY STATISTICS")
print("=" * 50)
print(f"Total Number of Classes : {len(class_counts)}")
print(f"Total Number of Images  : {total_images:,}")
print(f"Average per Class       : {np.mean(counts_list):.1f} images")
print(f"Median per Class        : {np.median(counts_list):.1f} images")

min_class = min(class_counts, key=class_counts.get)
max_class = max(class_counts, key=class_counts.get)
print(f"Least Common Class      : Class {min_class} ({CLASS_NAMES[min_class]}) -> {class_counts[min_class]} images")
print(f"Most Common Class       : Class {max_class} ({CLASS_NAMES[max_class]}) -> {class_counts[max_class]} images")
print(f"Imbalance Ratio (Max/Min): {class_counts[max_class] / max(1, class_counts[min_class]):.1f}x")
print("=" * 50)

### 📉 Step 4.3: Visualize Class Distribution (Detecting Class Imbalance)
In real life, drivers see Speed Limit 50 or Stop signs much more often than End of All Limits. GTSRB reflects this real-world distribution.
Let's plot a bar chart of all 43 classes to clearly visualize this distribution.

In [ ]:
plt.figure(figsize=(15, 6))
bars = plt.bar(range(43), [class_counts[i] for i in range(43)], color='#38bdf8', edgecolor='#0284c7', alpha=0.85)

# Highlight the minimum and maximum classes with distinct colors
bars[min_class].set_color('#f87171')  # Red for smallest
bars[max_class].set_color('#4ade80')  # Green for largest

plt.axhline(np.mean(counts_list), color='#f59e0b', linestyle='--', linewidth=1.5, label=f'Mean Count ({np.mean(counts_list):.0f})')

plt.title('Distribution of Images per Traffic Sign Class in GTSRB', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Class ID (0 to 42)', fontsize=11)
plt.ylabel('Number of Images', fontsize=11)
plt.xticks(range(0, 43, 2))
plt.grid(axis='y', linestyle=':', alpha=0.6)
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

print("💡 Viva Insight: This noticeable variation is known as 'Class Imbalance'.")
print("   During training, we use stratified train/test splitting so both sets receive balanced proportions of every class!")

### 📐 Step 4.4: Inspect Original Image Resolutions & Aspect Ratios
Traffic sign images in GTSRB are cropped from real video frames at different camera distances.
Let's inspect the dimensions of a random sample of images.

In [ ]:
sample_indices = np.random.choice(len(all_image_paths), size=min(500, len(all_image_paths)), replace=False)
widths = []
heights = []

for idx in sample_indices:
    _, fpath = all_image_paths[idx]
    img = cv2.imread(fpath)
    if img is not None:
        h, w, _ = img.shape
        widths.append(w)
        heights.append(h)

print(f"Inspected {len(widths)} sample images:")
print(f"  • Width  : Min = {min(widths)} px, Max = {max(widths)} px, Mean = {np.mean(widths):.1f} px")
print(f"  • Height : Min = {min(heights)} px, Max = {max(heights)} px, Mean = {np.mean(heights):.1f} px")

# Scatter plot of resolutions
plt.figure(figsize=(7, 5))
plt.scatter(widths, heights, color='#818cf8', alpha=0.5, s=25, edgecolors='none')
plt.plot([0, max(widths)], [0, max(widths)], color='#ef4444', linestyle='--', label='1:1 Square Ratio')
plt.title('Original Image Resolutions (Sample of 500 Images)', fontsize=12)
plt.xlabel('Width (pixels)')
plt.ylabel('Height (pixels)')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

print("💡 Takeaway: While aspect ratios are mostly square (around 1:1), resolutions vary widely (from ~25px to >200px).")
print("   Resizing all images to 32x32 pixels in Phase 5 provides a uniform tensor shape for the CNN.")

### 🖼️ Step 4.5: Visual Gallery of All 43 Traffic Sign Classes
Let's create a $6 \times 8$ subplot grid showing one sample from every single class.

In [ ]:
plt.figure(figsize=(18, 14))

for cid in range(43):
    plt.subplot(6, 8, cid + 1)
    if cid in sample_files:
        img = cv2.imread(sample_files[cid])
        if img is not None:
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            plt.imshow(img_rgb)
    
    # Clean, short title
    sign_name = CLASS_NAMES[cid]
    if len(sign_name) > 20:
        sign_name = sign_name[:18] + ".."
    plt.title(f"[{cid}] {sign_name}", fontsize=8.5)
    plt.axis('off')

plt.tight_layout()
plt.show()

### 🔍 Step 4.6: Real-World Variations (Lighting, Blur, Angles)
Traffic sign recognition in the wild is tricky because signs are seen under strong sunlight, night time, fog, rain, or motion blur.
Let's look at 8 different pictures of the **same sign** (e.g., Class 2: Speed Limit 50 km/h) to see how appearance varies.

In [ ]:
target_class = 2  # Speed Limit 50 km/h
target_folder = os.path.join(DATASET_PATH, str(target_class))

if os.path.exists(target_folder):
    class_files = os.listdir(target_folder)
    # Select 8 images spanning across different lighting and capture conditions
    step = max(1, len(class_files) // 8)
    chosen_files = [class_files[i * step] for i in range(min(8, len(class_files)))]
    
    plt.figure(figsize=(14, 3))
    for i, fname in enumerate(chosen_files):
        img_path = os.path.join(target_folder, fname)
        img = cv2.imread(img_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        plt.subplot(1, 8, i + 1)
        plt.imshow(img_rgb)
        plt.title(f"Sample {i+1}\n{img.shape[1]}x{img.shape[0]}", fontsize=8.5)
        plt.axis('off')
        
    plt.suptitle(f"Real-World Variations within Class {target_class}: {CLASS_NAMES[target_class]}", fontsize=12, y=1.05)
    plt.tight_layout()
    plt.show()
    print("💡 Notice how contrast, darkness, blur, and scale change dramatically across frames.")

## 🏆 Phase 4 Summary & Next Step
In Phase 4, we have:
- [x] Counted and verified images across all 43 GTSRB classes
- [x] Documented class imbalance (from ~200 to ~2,000 images per class)
- [x] Analyzed original image dimensions and aspect ratios
- [x] Generated a full visual catalog of all 43 traffic sign types
- [x] Visualized real-world capture variations (darkness, blur, camera distance)

👉 **You are now ready for Phase 5: Image Preprocessing Pipeline (`05_phase5_preprocessing_pipeline.ipynb`)!**